## Data Preparation for Splink Datasets

Preparing your datasets appropriately is crucial for effective record linkage using Splink. This guide demonstrates essential data cleaning and standardization steps to ensure your datasets are ready for analysis.

### 1. Import Necessary Libraries
Begin by importing the required libraries:

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import re

### 2. Set Display Options
Configure pandas to display all columns without line wrapping:

In [2]:
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.expand_frame_repr', False)  # Disable line wrapping

### 3. Create Sample Datasets
Define two sample datasets, data_a and data_b, to simulate real-world data scenarios:

In [3]:
# Dataset A
data_a = {
    'unique_id': [1, 2, 3, 4],
    'first_name': [' Mr. John ', 'Alice', ' Bob', ' Jane'],
    'surname': ['', ' Smith', 'Brown ', "O'Rama"],
    'date_of_birth': ['1980-01-01', '1990-05-15', ' 1985-07-20 ', '1975-09-10'],
    'city': [' New York', 'Los Angeles ', 'Chicago', 'San Francisco'],
    'email': [' john.doe@example.com ', 'alice.smith@example.com', 'bob.brown@example.com ', 'jane.ohara@example.com'],
    'address': ['123 Maple St.', '456 Oak Street', '789 Pine st.', '101 Birch Ave.']
}

# Dataset B
data_b = {
    'unique_id': [5, 6, 7, 8],
    'first_name': ['Mrs. Jon', ' Ms. Alicia ', 'Robert', 'Patrick'],
    'surname': ['Doe ', '', ' Brown', "O'Connor"],
    'dob': ['01/01/1980', '15-05-1990', '1985/07/20', '22.11.1982'],
    'location': ['NYC ', ' Los Angeles', 'Chicago', 'Boston'],
    'email_address': [' jon.d@example.com', 'alicia.s@example.com ', 'robert.b@example.com', 'patrick.oconnor@example.com'],
    'address': ['202 Elm St.', '303 Cedar Blvd.', '404 Spruce st.', '505 Willow St.']
}

df_a = pd.DataFrame(data_a)
df_b = pd.DataFrame(data_b)

### 4. Standardize Column Names
Ensure both datasets have consistent column names:

In [4]:
df_b.rename(columns={'dob': 'date_of_birth', 'location': 'city', 'email_address': 'email'}, inplace=True)

### 5. Trim Whitespace
Remove leading and trailing whitespace from all string entries:

In [5]:
df_a = df_a.map(lambda x: x.strip() if isinstance(x, str) else x)
df_b = df_b.map(lambda x: x.strip() if isinstance(x, str) else x)

### 6. Parse and Standardize Dates
Convert date strings to a uniform 'YYYY-MM-DD' format:

In [6]:
def parse_date(date_str):
    date_formats = ['%Y-%m-%d', '%Y/%m/%d', '%Y.%m.%d', '%d-%m-%Y', '%d/%m/%Y', '%d.%m.%Y', '%d %B %Y', '%B %d, %Y']
    for fmt in date_formats:
        try:
            return datetime.strptime(date_str, fmt)
        except ValueError:
            continue
    return pd.NaT

df_a['date_of_birth'] = df_a['date_of_birth'].apply(parse_date).dt.strftime('%Y-%m-%d')
df_b['date_of_birth'] = df_b['date_of_birth'].apply(parse_date).dt.strftime('%Y-%m-%d')

### 7. Convert Text to Lowercase
Standardize text data by converting to lowercase:

In [7]:
text_columns = ['first_name', 'surname', 'city', 'email', 'address']
for col in text_columns:
    df_a[col] = df_a[col].str.lower()
    df_b[col] = df_b[col].str.lower()

### 8. Standardize City Names
Ensure city names are consistent across datasets:

In [8]:
city_mapping = {'nyc': 'new york', 'los angeles': 'los angeles', 'chicago': 'chicago'}
df_a['city'] = df_a['city'].map(city_mapping).fillna(df_a['city'])
df_b['city'] = df_b['city'].map(city_mapping).fillna(df_b['city'])

### 9. Expand Common Abbreviations
Replace common abbreviations in address fields with their full forms:

In [9]:
abbreviations = {
    r'\bst\b\.?': 'street',
    r'\bave\b\.?': 'avenue',
    r'\brd\b\.?': 'road',
    r'\bdr\b\.?': 'drive',
    r'\bblvd\b\.?': 'boulevard',
    r'\bln\b\.?': 'lane',
    r'\bhwy\b\.?': 'highway',
    r'\bpkwy\b\.?': 'parkway',
    r'\bct\b\.?': 'court',
    r'\bpl\b\.?': 'place',
    r'\bsq\b\.?': 'square',
    r'\bmt\b\.?': 'mount',
    r'\bft\b\.?': 'fort'
}

for abbr, full in abbreviations.items():
    df_a['address'] = df_a['address'].replace(abbr, full, regex=True)
    df_b['address'] = df_b['address'].replace(abbr, full, regex=True)

### 10. Remove Honorifics from Names
Eliminate common honorifics from the first_name column:

In [10]:
honorifics_pattern = r'^\s*(?:mr|mrs|ms|dr)\.?\s+'

def remove_honorifics(name):
    return re.sub(honorifics_pattern, '', name, flags=re.IGNORECASE)

df_a['first_name'] = df_a['first_name'].apply(remove_honorifics)
df_b['first_name'] = df_b['first_name'].apply(remove_honorifics)

### 11. Remove special characters from string values
To ensure uniformity and prevent potential mismatches during data linkage:

In [11]:
# Remove single quotation marks from all string columns in DataFrame A
df_a = df_a.map(lambda x: x.replace("'", "") if isinstance(x, str) else x)

# Remove single quotation marks from all string columns in DataFrame B
df_b = df_b.map(lambda x: x.replace("'", "") if isinstance(x, str) else x)

### 12. Handle Null Values
Replace empty strings with NaN to ensure true null representation:

In [12]:
df_a.replace('', np.nan, inplace=True)
df_b.replace('', np.nan, inplace=True)

In [13]:
# Display the results
print("DataFrame A:")
print(df_a)
print("\nDataFrame B:")
print(df_b)

DataFrame A:
   unique_id first_name surname date_of_birth           city                    email           address
0          1       john     NaN    1980-01-01       new york     john.doe@example.com  123 maple street
1          2      alice   smith    1990-05-15    los angeles  alice.smith@example.com    456 oak street
2          3        bob   brown    1985-07-20        chicago    bob.brown@example.com   789 pine street
3          4       jane   orama    1975-09-10  san francisco   jane.ohara@example.com  101 birch avenue

DataFrame B:
   unique_id first_name  surname date_of_birth         city                        email              address
0          5        jon      doe    1980-01-01     new york            jon.d@example.com       202 elm street
1          6     alicia      NaN    1990-05-15  los angeles         alicia.s@example.com  303 cedar boulevard
2          7     robert    brown    1985-07-20      chicago         robert.b@example.com    404 spruce street
3          8 

When saving a pandas DataFrame to a CSV file using the **to_csv()** method, missing values (**NaN**) are typically represented as empty fields in the resulting CSV. This default behavior can lead to ambiguity when distinguishing between actual empty strings and missing values. To explicitly represent this missing values in CSV file, we can utilize the **na_rep** parameter of the **to_csv()** method to specify a placeholder for **NaN** values.

The **na_rep='NULL'** parameter ensures that all **NaN** values in the DataFrame are written as 'NULL' in the CSV file. 

Select a placeholder (na_rep value) that does not conflict with actual data entries to avoid misinterpretation. Common choices include 'NULL', 'NA', or any string that is not present in our data.

In [14]:
# Save df_a to a CSV file
df_a.to_csv('../../data/cleaned_dataset_a.csv', index=False, na_rep='NULL')

# Save df_b to a CSV file
df_b.to_csv('../../data/cleaned_dataset_b.csv', index=False, na_rep='NULL')